In [ ]:
!pip install -q openai langchain langchain-openai langchain-core langchain-community langchain-anthropic

In [ ]:
!pip install arxiv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [
           {"role":"user", "content":"what is the GDP of india 2024-25?"}
    ]
)
print(response.choices[0].message.content)

It is difficult to accurately predict the GDP of India for the year 2024-25 as it depends on various factors such as economic growth, government policies, and global economic conditions. The GDP forecast for India in 2024-25 will be available closer to the time period.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
response = llm.invoke("what is langchain explain in esingle sentence")
print(response.content)
print(f"Token Usage {response.response_metadata.get('token_usage','N/A')}")

Langchain is a decentralized language learning platform that uses blockchain technology to connect language learners with native speakers for personalized language exchange.
Token Usage {'completion_tokens': 24, 'prompt_tokens': 16, 'total_tokens': 40, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}


In [ ]:
#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

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt | model | parser  #LCEL = Langchain Expression Language
result = chain.invoke({"topic":"cats"})
print(result)

Why was the cat sitting on the computer?

Because it wanted to keep an eye on the mouse!


In [ ]:
messages = [
    SystemMessage(content="You are a nice AI bot that helps a user figure out what to eat in one short sentence"),
    HumanMessage(content="I like tomatoes, what should I eat?")
]
response = llm.invoke(messages)
print(response.content)

You could try a caprese salad with fresh tomatoes, mozzarella, basil, and balsamic glaze.


In [ ]:
conversation=[
    SystemMessage(content="You are a math tutor.Be brief"),
    HumanMessage(content="What is a derivative?"),
    AIMessage(content="A derivative measures the rate of change of a function at any point.It's the slope of the tangent line"),
    HumanMessage(content="Give me a real-world example of that")
]

response = llm.invoke(conversation)
print(response.content)

The speedometer in a car measures the rate of change of distance with respect to time, which is essentially the derivative of the distance function.


In [ ]:
#tool calling
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

#create tool
arxiv_wrapper = ArxivAPIWrapper(
     top_k = 3,
     doc_content_chars_max=1000
)
arxiv_tool = ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)
print(f"Tool name   {arxiv_tool.name}")
print(f"Tool description   {arxiv_tool.description}")

Tool name   arxiv
Tool description   A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.


In [ ]:
results = arxiv_tool.invoke("attentio is all you need transformer")
print(f"Archiv results :\n {results[:800]}")

Archiv results :
 Published: 2025-12-03
Title: "All You Need" is Not All You Need for a Paper Title: On the Origins of a Scientific Meme
Authors: Anton Alyakin
Summary: The 2017 paper ''Attention Is All You Need'' introduced the Transformer architecture-and inadvertently spawned one of machine learning's most persistent naming conventions. We analyze 717 arXiv preprints containing ''All You Need'' in their titles (2009-2025), finding exponential growth ($R^2$ > 0.994) following the original paper, with 200 titles in 2025 alone. Among papers following the canonical ''X [Is] All You Need'' structure, ''Attention'' remains the most frequently claimed necessity (28 occurrences). Situating this phenomenon within memetic theory, we argue the pattern's success reflects competitive pressures in scientific communica


In [ ]:
from langchain_core.tools import tool
@tool
def caclulator(expression:str) -> str:
  """Calculate a mathematical expression.Use for any amth operation.
  Example inputs '2 + 2','100*15','(2+3)/4' """
  return str(eval(expression))
@tool
def word_count(text:str) -> str:
  """ Count the number if words in given text."""
  count = len(text.split())
  return f"The count of words is {count}"
tools=[arxiv_tool,caclulator,word_count]
llm_with_tools = llm.bind_tools(tools)
response = llm_with_tools.invoke("what is 15% of 2847?")
for tc in  response.tool_calls:
  print(f" Tool :{tc['name']},Args{tc['args']}")

 Tool :caclulator,Args{'expression': '2847*15/100'}
